# TP 2.1 — GAN on Lego Brick Images
**Introduction to Deep Generative Models — Arthur SATOUF**  

For each practical exercise (TP), work in groups of two or three. Create a **private GitHub repository**, add **arthur-75**.

**Goal:** Train a DCGAN (Deep Convolutional GAN) to generate **64 × 64 grayscale images of Lego bricks**.

> Complete the `xxx`  placeholders and unfinished functions yourself. Some cells are intentionally not executable until completed.

## 1. Understanding the Dataset

- Data: A Kaggle dataset containing PNG/JPG images of Lego bricks.
- Load images from a folder (labels are not needed).
- Convert to grayscale (one channel), resize to 64 × 64, and normalize.

**Question:** Should you normalize pixels to **[-1, 1]** or **[0, 1]** here, and why?

In [ ]:
%pip install kaggle
!kaggle datasets download -d joosthazelzet/lego-brick-images --unzip

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.utils as vutils
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# --------------------
# Hyperparameters
# --------------------
IMAGE_SIZE = 64
CHANNELS = 1
BATCH_SIZE = 128
Z_DIM = 100
EPOCHS = 100
LEARNING_RATE = 0.0002
BETA1 = 0.5
BETA2 = 0.999
DEVICE = "mps"  # or "cuda" if torch.cuda.is_available() else "cpu"
NOISE_PARAM = 0.1

# --------------------
# Transforms
# --------------------
# Grayscale + Resize + Normalize
transform = xxx.xxx([
    xxxx,  # Grayscale
    xxx,   # Resize
    xxx.ToTensor(),
    xxx    # Center around 0 with range [-1, 1]
])

### 1.1 Build Dataset & DataLoader

Create a custom dataset returning only the images from a specified folder.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.xxx = xxx
        self.xxx = xxx
        # Gather .png, .jpg and .jpeg images
        self.image_files = sorted([
            f for f in os.listdir(root_dir)
            if f.endswith(('.png', '.jpg', '.jpeg'))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert("RGB")  # or "L" for grayscale
        if self.transform:
            image = self.transform(image)
        return image

# Specify the dataset folder path
dataset_path = "dataset"
dataset = ImageDataset(xxx=xxx, xxx=xxx)
dataloader = DataLoader(xxx, batch_size=xxx, shuffle=True)

### 1.2 Plot a few examples

Implement a function to visualize example images from the DataLoader.

In [ ]:
def plot_sample_images(loader):
    xxx

plot_sample_images(dataloader)

## 2. Building the GAN Architecture

### 2.1 Discriminator

- **Input:** A 64 × 64 image, real or generated.
- **Goal:** Produce a single value representing whether the image is real.
- Use `Conv2d` to downsample from 64 × 64 to 1 × 1.
- Use `LeakyReLU`, an output `Sigmoid`, and intermediate BatchNorm layers.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            # Input shape: (CHANNELS, 64, 64)
            nn.Conv2d(CHANNELS, xxx, 4, 2, 1, bias=False),
            nn.xxx(0.2, inplace=True),  # activation
            nn.xxx(0.3),  # dropout

            nn.xxx(xxx, 128, 4, 2, 1, bias=False),  # conv
            nn.xxx(xxx),  # batch norm
            nn.xxx(0.2, inplace=True),
            nn.xxx(0.3),

            nn.xxx(xxx, 256, 4, 2, 1, bias=False),
            nn.xxx(xxx),
            nn.xxx(0.2, inplace=True),
            nn.xxx(0.3),

            nn.xxx(256, xxx, 4, 2, 1, bias=False),
            nn.xxx(512),
            nn.xxx(0.2, inplace=True),
            nn.xxx(0.3),

            # Output: (1, 1, 1), then flatten to scalar
            nn.xxx(512, 1, 4, 1, 0, bias=False),
            nn.xxx(),  # activation [0, 1]
            nn.xxx(start_dim=1)
        )

    def forward(self, x):
        return self.model(x)

### 2.2 Generator

- **Input:** Random noise `z` of dimension `Z_DIM` (e.g., 100).
- **Goal:** Generate a 64 × 64 grayscale image.
- Use `ConvTranspose2d` to upsample from 1 × 1 to 64 × 64.
- Use `LeakyReLU` between layers and `Tanh` for output in **[-1, 1]**.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            # Input: (Z_DIM, 1, 1)
            nn.ConvTranspose2d(Z_DIM, xxx, 4, 1, 0, bias=False),
            nn.BatchNorm2d(xxx),
            nn.LeakyReLU(0.2, inplace=True),

            nn.xxx(xxx, xxx, 4, 2, 1, bias=False),
            nn.xxx(xxx),
            nn.xxx(0.2, inplace=True),

            nn.xxx(xxx, xxx, 4, 2, 1, bias=False),
            nn.xxx(xxx),
            nn.xxx(0.2, inplace=True),

            nn.xxx(xxx, xxx, 4, 2, 1, bias=False),
            nn.xxx(xxx),
            nn.xxx(0.2, inplace=True),

            nn.xxx(xxx, CHANNELS, 4, 2, 1, bias=False),
            nn.xxx()  # activation [-1, 1]
            # Output: (CHANNELS, 64, 64)
        )

    def forward(self, x):
        return self.model(x)

# Instantiate models
critic = Discriminator().to(DEVICE)
generator = Generator().to(DEVICE)

## 3. Training Setup

1. **Loss:** Binary Cross-Entropy for both generator and discriminator. Real images have label 1; fake images label 0.
2. **Optimizers:** Adam with `betas=(0.5, 0.999)`.
3. **Optional:** Add small noise to the labels so the discriminator does not overpower the generator.

In [ ]:
# Losses & Optimizers
criterion = nn.xxx()  # Binary Cross Entropy
d_optimizer = xxx.xxx(xxx.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2))
g_optimizer = optim.Adam(xxx.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2))

### 3.1 Training Loop

Alternate between discriminator and generator updates. Keep the placeholders intact until you implement the missing operations.

In [ ]:
g_loss_values = []
d_loss_values = []
g_acc_values = []
d_acc_values = []

for epoch in range(EPOCHS):
    g_loss_batch = []
    d_loss_batch = []
    g_acc_batch = []
    d_acc_batch = []

    progress_bar = tqdm(enumerate(dataloader), total=len(dataloader),
                        desc=f"Epoch {epoch+1}/{EPOCHS}")
    for i, real_images in progress_bar:
        real_images = real_images.to(DEVICE)
        batch_size = real_images.size(0)

        # 1) Train Discriminator
        xxx.xxx()

        # True labels near 1, with noise
        real_labels = torch.xxx(xxx, 1, device=DEVICE)
        real_noisy_labels = real_labels - NOISE_PARAM * torch.rand_like(real_labels)
        # Fake labels near 0, with noise
        fake_labels = torch.xxx(xxx, 1, device=DEVICE)
        fake_noisy_labels = fake_labels + NOISE_PARAM * torch.rand_like(fake_labels)

        real_preds = xxx(real_images)
        real_loss = xxx(real_preds, real_noisy_labels)

        z = torch.randn(batch_size, Z_DIM, 1, 1, device=DEVICE)
        fake_images = xxx(z)
        fake_preds = xxx(fake_images.detach())
        fake_loss = xxx(fake_preds, fake_noisy_labels)

        d_loss = (xxx + xxx) / 2
        d_loss.xxx()
        d_optimizer.xxx()

        # Discriminator accuracy
        d_real_acc = (real_preds.round() == real_labels).float().mean().item()
        d_fake_acc = (fake_preds.round() == fake_labels).float().mean().item()
        d_acc = (d_real_acc + d_fake_acc) / 2

        # 2) Train Generator
        xxx.xxx()
        fake_preds = xxx(fake_images)  # do not detach here
        # Generator wants the discriminator to predict "real"
        g_loss = xxx(fake_preds, real_labels)
        xxx.xxx()
        xxx.xxx()

        g_acc = (fake_preds.round() == real_labels).float().mean().item()

        d_loss_batch.append(d_loss.item())
        g_loss_batch.append(g_loss.item())
        d_acc_batch.append(d_acc)
        g_acc_batch.append(g_acc)
        progress_bar.set_postfix(D_Loss=d_loss.item(), G_Loss=g_loss.item())

    # End of epoch
    d_loss_values.append(np.mean(d_loss_batch))
    g_loss_values.append(np.mean(g_loss_batch))
    d_acc_values.append(np.mean(d_acc_batch))
    g_acc_values.append(np.mean(g_acc_batch))

    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            z_sample = torch.randn(16, Z_DIM, 1, 1, device=DEVICE)
            fake_sample = generator(z_sample).cpu()
        vutils.save_image(fake_sample, f"fake_sample_epoch_{epoch+1:03d}.png",
                          normalize=True)

### 3.2 Optional: Load a pre-trained model

If training does not work, the exercise proposes loading pre-trained generator and discriminator weights. Provide the file paths first.

In [ ]:
# Load state dictionaries
# GEN_PATH and CRITIC_PATH must be supplied
generator.load_state_dict(torch.load(GEN_PATH, map_location=DEVICE))
critic.load_state_dict(torch.load(CRITIC_PATH, map_location=DEVICE))
generator.eval()
critic.eval()

## 5. Monitoring Progress

- Plot discriminator and generator losses.
- Optionally, plot discriminator and generator accuracy.
- Save or visualize generated images to track their evolution.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

titles = ['epoch_g_loss', 'epoch_g_acc', 'epoch_d_loss', 'epoch_d_acc']
data = [g_loss_values, g_acc_values, d_loss_values, d_acc_values]

for ax, title, values in zip(axes, titles, data):
    ax.plot(values, color='brown')
    ax.set_title(title)
    ax.grid(True)

plt.tight_layout()
plt.show()

**Questions**

- What do you observe? What's wrong?
- If the discriminator loss is near zero but the generator loss is large, what does that imply?
- What does **mode collapse** look like?

## 6. Generating and Evaluating

1. Sample random latent vectors `z` and generate images.
2. Save generated images as `.png` files or show them in a grid.
3. *(Optional)* Compare generated images with real dataset images for diversity.

In [ ]:
# Sampling: Generate a grid of images from random noise
grid_width, grid_height = (8, 4)
z_sample = torch.xxx(xxx * xxx, xxx, 1, 1, device=DEVICE)
gen_imgs = xxx(z_sample).cpu().detach()

fig = plt.figure(figsize=(grid_width, grid_height))
for i in range(grid_width * grid_height):
    ax = fig.add_subplot(grid_height, grid_width, i + 1)
    ax.axis("off")
    # Each image shape: (1, 64, 64)
    ax.imshow(gen_imgs[i, 0, :, :], cmap="gray")
plt.show()